In [ ]:
import sys
import pandas as pd
sys.path.insert(0, "/Users/carang/Documents/pycharm document projects/SpotifyProj/docs")
from spotify_loading import load_history, align_end_dates, filter_window

data = "/Users/carang/Documents/pycharm document projects/SpotifyProj/data/"

#fetch the data for both people
cara = load_history(data + "my_spotify_data2 copy.zip", "Cara")
ruby = load_history(data + "my_spotify_datahenkek.zip", "Ruby")

#concatenate both peoples data
both_clean = align_end_dates(pd.concat([cara, ruby], ignore_index=True))

# pick a window
window = input("Select a time frame (all, 30d, 6m, 12m, year): ").strip().lower()

if window == "year":
    year = int(input("Which year? "))
    window_df = filter_window(both_clean, "year", year)
else:
    window_df = filter_window(both_clean, window)

print("now in window:" + window)


In [ ]:
from artists_tab import artist_report
#run the block below before running this lmao

artist_input = input("Please enter the name of the artist: ")
result = artist_report(window_df, artist_input, "Cara", "Ruby")   # use a real artist
show_artist(result, "Cara", "Ruby")

In [ ]:
#show the individual artist search in a better format
import matplotlib.pyplot as plt

def show_artist(result, person_a, person_b):
    if result is None:
        print("Artist not found.")
        return

    print(f"===== {result['artist']} =====\n")

    for person in [person_a, person_b]:
        stats = result[person]
        if stats is None:
            print(f"{person} has never listened to this artist.\n")
            continue

        print(f"{person}:")
        print(f"  First listened:  {stats['first_date'].strftime('%b %d, %Y')}")
        print(f"  First song:      {stats['first_song']}")
        print(f"  Hours:           {stats['hours']:.1f}")
        print(f"  Rank:            #{stats['rank']}")
        print(f"  Top songs:")
        for song, plays in stats["top_songs"].items():
            print(f"    - {song} ({plays} plays)")
        print()

    if "listened_first" in result:
        print(f"🎧 {result['listened_first']} listened first, by {result['gap_days']} days")
        print(f"🏆 Bigger fan: {result['bigger_fan']}")

    #plotting it
    timeline = pd.DataFrame(result["timeline"])
    plt.figure(figsize=(12, 5))
    plt.plot(timeline.index, timeline["Cara"], label="Cara")
    plt.plot(timeline.index, timeline["Ruby"], label="Ruby")
    plt.title(f"Listening over time: {result['artist']}")
    plt.xlabel("Month")
    plt.ylabel("Hours")
    plt.xticks(rotation=45)  # tilt month labels so they don't overlap
    plt.legend()  # shows which line is who
    plt.tight_layout()  # stops labels getting cut off
    plt.show()


In [ ]:
#individual top artists
from artists_tab import show_individual_artists_ranking

table_A, table_B = show_individual_artists_ranking(window_df, "Cara", "Ruby")

print(table_A.head(20))
print(table_B.head(20))

In [ ]:
#shared top artists
from artists_tab import show_shared_artists_ranking

table = show_shared_artists_ranking(window_df, "Cara", "Ruby")
print(table.head(20))